# 00 Optional Input Manifest

**Methods mapping:** preflight only; not a manuscript Methods subsection.

This notebook records the input files used by the analysis: metadata and depth-QC tables, reference files, the full LoFreq raw iSNV CSV, TF003/T10 G-gene haplotype FASTA files, study consensus G FASTA files, and the global G FASTA file. It records file presence, hashes, row counts, and FASTA sequence counts.


In [1]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / 'config' / 'analysis_config.yaml').exists():
        ROOT = candidate
        break
sys.path.insert(0, str(ROOT / 'notebooks'))
import analysis_utils as au
pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 160)
rel = lambda path: Path(path).relative_to(ROOT).as_posix()
ROOT


PosixPath('/root/autodl-tmp/rsva-g-intrahost-haplotypes')

In [2]:
DATA_DIR, FIG_DIR = au.step_dirs('00_input_manifest', ROOT)
INPUTS = {
    "metadata": ROOT / "data/metadata/meta_v6_with_season_clade.csv",
    "depth_qc": ROOT / "data/metadata/qualified_samples_depth200_pos90.csv",
    "reference_fasta": ROOT / "data/reference/reference.fasta",
    "annotation_gff": ROOT / "data/reference/genome_annotation.gff",
    "g_reference_alignment": ROOT / "data/reference/EPI_ISL_412866_G_4652-5617_reference_alignment.fasta",
    "lofreq_raw_calls": ROOT / "data/input/isnv/lofreq_raw_calls.csv",
    "AU_haplotype_fasta": ROOT / "data/input/haplotypes/PRJNA1037681_extracted_4652-5617.fasta",
    "US_haplotype_fasta": ROOT / "data/input/haplotypes/PRJNA1130896_extracted_4652-5617.fasta",
    "AU_consensus_fasta": ROOT / "data/input/consensus/PRJNA1037681_all_consensus_extracted_4652-5617.fasta",
    "US_consensus_fasta": ROOT / "data/input/consensus/PRJNA1130896_all_consensus_extracted_4652-5617.fasta",
    "global_g_fasta": ROOT / "data/input/global/nextstrain_2026_global_G_4652-5617_filtered.fasta",
}
OUTPUTS = {
    "input_manifest": DATA_DIR / "input_file_manifest.csv",
    "manifest_summary": DATA_DIR / "input_file_manifest_summary.csv",
    "missing_inputs": DATA_DIR / "missing_required_inputs.csv",
}


def show_paths(title, paths):
    rows = []
    for name, path in paths.items():
        path = Path(path)
        rows.append({"name": name, "relative_path": rel(path), "exists": path.exists()})
    display(Markdown(f"### {title}"))
    display(pd.DataFrame(rows))

for path in OUTPUTS.values():
    Path(path).parent.mkdir(parents=True, exist_ok=True)

show_paths("Input paths", INPUTS)
show_paths("Output paths", OUTPUTS)
DATA_DIR, FIG_DIR


### Input paths

,name,relative_path,exists
0,metadata,data/metadata/meta_v6_with_season_clade.csv,True
1,depth_qc,data/metadata/qualified_samples_depth200_pos90...,True
2,reference_fasta,data/reference/reference.fasta,True
3,annotation_gff,data/reference/genome_annotation.gff,True
4,g_reference_alignment,data/reference/EPI_ISL_412866_G_4652-5617_refe...,True
5,lofreq_raw_calls,data/input/isnv/lofreq_raw_calls.csv,True
6,AU_haplotype_fasta,data/input/haplotypes/PRJNA1037681_...,True
7,US_haplotype_fasta,data/input/haplotypes/PRJNA1130896_...,True
8,AU_consensus_fasta,data/input/consensus/PRJNA1037681_a...,True
9,US_consensus_fasta,data/input/consensus/PRJNA1130896_a...,True


### Output paths

,name,relative_path,exists
0,input_manifest,data/processed_data/00_input_manifest/in...,True
1,manifest_summary,data/processed_data/00_input_manifest/in...,True
2,missing_inputs,data/processed_data/00_input_manifest/mi...,False


(PosixPath('/root/autodl-tmp/rsva-g-intrahost-haplotypes/data/processed_data/00_input_manifest'),
 PosixPath('/root/autodl-tmp/rsva-g-intrahost-haplotypes/results/figures/00_input_manifest'))

## Input File Manifest

The manifest below is written to `data/processed_data/00_input_manifest/`. A missing file here should stop the analysis before downstream notebooks are run.


In [3]:
manifest, summary = au.input_manifest(ROOT, DATA_DIR)
manifest_view = manifest[['role', 'path', 'exists', 'rows', 'columns', 'fasta_sequences', 'bytes', 'sha256']]

display(summary)
display(manifest_view)
missing = manifest_view.loc[~manifest_view['exists'], ['role', 'path']]
if len(missing):
    display(Markdown('### Missing required inputs'))
    display(missing)
else:
    display(Markdown('All listed input files are present.'))


,exists,files,total_bytes
0,True,11,18711894


,role,path,exists,rows,columns,fasta_sequences,bytes,sha256
0,metadata,data/metadata/meta_v6_with_season_clade.csv,True,521.0,62.0,NaN,224123,4588ea24845789f937472e8c1f5886d1c103132036a15a...
1,depth_qc,data/metadata/qualified_samples_depth200_pos90...,True,572.0,6.0,NaN,77334,de12feeee638d5292072e5916f6d2da1541607950b75e5...
2,reference,data/reference/reference.fasta,True,NaN,NaN,1.0,15242,785f6ea7847f2fd6b43a3ab9fc1f0e689e2045961d5a88...
3,annotation,data/reference/genome_annotation.gff,True,NaN,NaN,NaN,2019,f4d5a17b4036c6aacae770d3fce36d1b5b06260ebdb76d...
4,G_reference_alignment,data/reference/EPI_ISL_412866_G_4652-5617_refe...,True,NaN,NaN,1.0,1007,152ee47d9d6dad9461ad8f6a06a4efdad8c4beecd91df3...
5,LoFreq_raw_calls,data/input/isnv/lofreq_raw_calls.csv,True,51439.0,22.0,NaN,5781371,6b09b56514a5c4d94099b2e9b1590f9e05fc28f2d30b89...
6,global_G_consensus_FASTA,data/input/global/nextstrain_2026_g...,True,NaN,NaN,12382.0,12270562,00c15db2d31e9f7b21d2f0620d20e0ebad2310cba1863c...
7,PRJNA1037681_haplotype_G_FASTA,data/input/haplotypes/PRJNA1037681_...,True,NaN,NaN,75.0,77273,28edbbbb6c24514645cd337ec8ccf940d68bd5f33183de...
8,PRJNA1037681_study_consensus_G_FASTA,data/input/consensus/PRJNA1037681_a...,True,NaN,NaN,38.0,37851,f9f8017a8d1bf0ba843741f90692cc784fd04eb7525a45...
9,PRJNA1130896_haplotype_G_FASTA,data/input/haplotypes/PRJNA1130896_...,True,NaN,NaN,145.0,149413,ff40875529860da3a8ad6114c0c0a6a16fb97e9ada1ddc...


All listed input files are present.